# AoC 2024 Day 2 — Red-Nosed Reports

**Spark lesson: higher-order array functions (no UDFs)**

Puzzle: <https://adventofcode.com/2024/day/2>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

---

## The puzzle

Each line is a *report*: a list of levels.

- **Part 1** — a report is **safe** when its levels are either all increasing or all decreasing, *and* every adjacent step is between 1 and 3 inclusive. Count the safe reports.
- **Part 2** — the Problem Dampener: a report also counts as safe if removing a single level would make it safe. Count those too.

## The Spark angle

The interesting constraint here is self-imposed: **solve it without a UDF.**

A Python UDF would serialise every row out to a Python worker and back, losing an order of magnitude and blinding the optimiser. Spark's higher-order functions — `transform`, `zip_with`, `forall`, `exists`, `slice`, `sequence` — run on array columns *inside the JVM*, so the whole thing stays one native expression tree.

- adjacent differences → `zip_with(levels[1:n-1], levels[2:n], (a,b) -> b-a)`
- "every step is legal" → `forall(diffs, d -> d >= 1 AND d <= 3)`
- "some single removal rescues it" → `exists(variants, ...)`, which short-circuits on the first success rather than building all *n* variants.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day02

spark = get_spark('aoc-2024-day02')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = """\
7 6 4 2 1
1 2 7 8 9
9 7 6 2 1
1 3 2 4 5
8 6 4 4 1
1 3 6 7 9
"""

print('part 1:', day02.part1(spark, EXAMPLE), '(expected 2)')
print('part 2:', day02.part2(spark, EXAMPLE), '(expected 4)')

### The expression tree, made visible

Every intermediate below is a *column*, not a Python value — nothing has been pulled to the driver yet.

In [ ]:
from pyspark.sql import functions as F

df = day02.parse(spark, EXAMPLE)
levels = F.col('levels')
size = F.size(levels)

df.select(
    'levels',
    F.zip_with(
        F.slice(levels, F.lit(1), size - 1),
        F.slice(levels, F.lit(2), size - 1),
        lambda a, b: b - a,
    ).alias('diffs'),
    day02._is_safe(levels).alias('safe'),
    F.exists(day02._dampened_variants(levels), day02._is_safe).alias('rescued'),
).show(truncate=False)

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 2)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

for part in (1, 2):
    fn = getattr(day02, f'part{part}')
    started = time.perf_counter()
    answer = fn(spark, data)
    print(f'part {part}: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Notes & gotchas

- `slice` is **1-based**, which is why `_dampened_variants` uses `i + 2` to skip element *i*. Off-by-one here is the classic bug — the example is small enough to check by eye.
- Look at the `rescued` column for row `7 6 4 2 1`: it is already safe, so part 2 must OR the two conditions rather than replacing one with the other.